# Modelle laden

Im vorherigen Notebook haben wir das große Modell gespeichert. In diesem Notebook sehen wir uns an, wie wir es neu laden können.

## Setup & Daten

In [ ]:
# Pakete importieren
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

tf.keras.backend.set_floatx("float64")

# Random Seed für das gesamte Notebook definieren
RSEED = 42

In [ ]:
# Daten laden
df = pd.read_csv("../data/boston.csv")

# Zielvariable definieren
y = df.pop("MEDV")

# In Trainings- und Testdatensatz aufteilen
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [ ]:
# Numerische Features skalieren
# Numerische Werte skalieren
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

In [ ]:
scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# In np-Array umwandeln
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

Das SavedModel-Format ist ein Verzeichnis mit einer Protobuf-Binärdatei und einem TensorFlow-Checkpoint. Sieh dir das Verzeichnis des gespeicherten Modells an:

> [!NOTE]
> Dieses Lösungsnotebook liegt im Ordner `solutions/`. Das Modell wird von Notebook 3 (`day_2/03_overfit_underfit.ipynb`) erstellt, das es unter `day_2/saved_model/` speichert. Deshalb greifen die Zellen unten über den relativen Pfad `../day_2/saved_model/my_large_model.keras` darauf zu.
>
> Führ zuerst Notebook 3 aus, damit das Modell existiert, sonst finden diese Zellen es nicht.

In [ ]:
!ls ../day_2/saved_model

In [ ]:
!ls ../day_2/saved_model/my_large_model.keras

## SavedModel-Format

Lade aus dem gespeicherten Modell ein frisches Keras-Modell neu:

In [ ]:
# Gespeichertes Modell laden
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("../day_2/saved_model/my_large_model.keras")

# Architektur des Modells prüfen
new_large_model.summary()

Das wiederhergestellte Modell ist mit denselben Argumenten kompiliert wie das ursprüngliche Modell. Versuch, mit dem geladenen Modell evaluate und predict auszuführen:

In [ ]:
# Wiederhergestelltes Modell evaluieren
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")

Der MSE des Modells ist höher als im vorherigen Notebook, obwohl wir dieselbe Aufteilung der Daten verwenden. Wir verwenden dasselbe Modell und dieselben Daten. Deshalb würden wir erwarten, auch denselben MSE zu erhalten. 

> **Aufgabe:** Findest du heraus, an welcher Stelle dieses Notebook vom Vorgehen im vorherigen Notebook abweicht? Es kann helfen, dir die Werte in X_train in beiden Notebooks anzusehen.

<details><summary>
Klick hier für einen Tipp ...
</summary>
Sieh dir an, wie die Daten vorverarbeitet wurden. Haben beide Notebooks denselben Scaler verwendet?
</details>

Das Modell und die Aufteilung in Trainings- und Testdaten sind dieselben wie im vorherigen Notebook. Das Einzige, was die Vorhersagen verändern kann, ist also, wie die Features vorverarbeitet wurden. Wir grenzen diesen Unterschied jetzt ein und beheben ihn.

### Schritt 1: beide Scaler neu erstellen

Das vorherige Notebook (`03_overfit_underfit`) hat die Feature-Spalten mit einem `StandardScaler` skaliert, dieses Notebook dagegen dieselben Spalten mit einem `MinMaxScaler`. Wir laden die Daten mit derselben Aufteilung neu und fitten beide Scaler auf den Trainingsspalten, damit wir sie direkt vergleichen können.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Neu laden und genau wie zuvor aufteilen
df_cmp = pd.read_csv("../data/boston.csv")
y_cmp = df_cmp.pop("MEDV")
X_train_cmp, X_test_cmp, y_train_cmp, y_test_cmp = train_test_split(
    df_cmp, y_cmp, random_state=RSEED
)

# Beide Scaler nur auf den Trainingsspalten fitten
minmax_scaler = MinMaxScaler().fit(X_train_cmp[col_scale])      
standard_scaler = StandardScaler().fit(X_train_cmp[col_scale])  # der Scaler, den NOTEBOOK 3 verwendet hat

### Schritt 2: skalierte Spalten vergleichen

Die beiden Scaler bilden dieselben Spalten auf sehr unterschiedliche Wertebereiche ab. `MinMaxScaler` presst jede Spalte in den Bereich 0 bis 1, sodass kein Wert negativ ist. `StandardScaler` zentriert jede Spalte um 0 und erzeugt deshalb auch negative Werte. Das Modell wurde mit der zweiten Art von Eingaben trainiert.

In [ ]:
minmax_values = minmax_scaler.transform(X_train_cmp[col_scale])
standard_values = standard_scaler.transform(X_train_cmp[col_scale])

print(f"MinMaxScaler   scaled columns range: {minmax_values.min():.2f} to {minmax_values.max():.2f}")
print(f"StandardScaler scaled columns range: {standard_values.min():.2f} to {standard_values.max():.2f}")

### Schritt 3: Modell mit dem passenden Scaler erneut evaluieren

Wenn der Scaler das Problem ist, sollte der MSE wieder auf den Wert aus Notebook 3 sinken, sobald wir den Testdatensatz mit `StandardScaler` skalieren (genau wie Notebook 3) und dasselbe geladene Modell evaluieren.

In [ ]:
# Den Testdatensatz so skalieren wie Notebook 3: StandardScaler auf denselben Spalten
X_test_std = X_test_cmp.copy()
X_test_std[col_scale] = standard_scaler.transform(X_test_cmp[col_scale])

# Dasselbe geladene Modell auf dem korrekt skalierten Testdatensatz evaluieren
with tf.device("/cpu:0"):
    loss_std, mse_std = new_large_model.evaluate(
        X_test_std.values, y_test_cmp.values, verbose=2
    )

print(f"MSE with MinMaxScaler (this notebook):        {mse:.2f}")
print(f"MSE with StandardScaler (matching notebook 3): {mse_std:.2f}")

### Antwort

Die beiden Notebooks verwenden **unterschiedliche Scaler**. Notebook 3 hat das Modell mit Features trainiert, die mit einem `StandardScaler` skaliert wurden (Mittelwert 0, Standardabweichung 1), dieses Notebook hat die Features dagegen mit einem `MinMaxScaler` skaliert (Bereich 0 bis 1). Ein neuronales Netz lernt Gewichte, die genau auf die Skala seiner Trainingseingaben zugeschnitten sind. Bekommt es Daten auf einer anderen Skala, verschiebt sich jede Vorhersage, und der MSE steigt.

Wenn wir den Testdatensatz mit demselben `StandardScaler` skalieren, sinkt der MSE wieder auf das Niveau von Notebook 3. Das bestätigt, dass der Scaler die Ursache war.

Die Vorverarbeitung ist Teil des Modells. Jede Transformation, die du beim Training fittest, muss bei der Vorhersage genau gleich angewendet werden. In der Praxis würdest du zusammen mit dem Modell **den gefitteten Scaler speichern** (zum Beispiel mit `joblib`) und ihn neu laden, statt einen neuen, anderen Scaler zu fitten.